# SkinVisionNet: Deep Learning Techniques for Accurate Pigmented Skin Lesion Classification

## 1. Import delle librerie

In [34]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from torchvision import transforms
from torch.utils.data import Subset
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from tqdm import tqdm
import torch


In [ ]:
CONFIG = {
    # Dataset
    'dataset_path': "ISIC_2019_Input",
    'groundtruth_path': "ISIC_2019_GroundTruth.csv",
    'train_size': 0.8,
    
    # Immagini 
    'image_size': 320,              
    'batch_size': 10,              
    
    # Modello
    'modello': "efficientnet_b4",
    'dropout_rate': 0.7,            
    'drop_path_rate': 0.3,       
    
    # Ottimizzazione
    'LR': 3e-5,                    
    'WD': 3e-3,                      
    'epoche': 25,                   
    'grad_clip_norm': 0.5,           
    'mixed_precision': True,
    
    # Loss function
    'loss_type': 'focal',
    'focal_alpha': 1.0,
    'focal_gamma': 2.0,
    
    # Scheduler
    'scheduler_type': 'cosine',
    'warmup_epochs': 3,           
    'min_lr': 1e-7,                 
    
    # Augmentazioni
    'aug_hflip_prob': 0.7,         
    'aug_vflip_prob': 0.7,          
    'aug_rotation': 60,             
    'aug_crop_scale': (0.6, 1.0),   
    'aug_brightness': 0.4,          
    'aug_contrast': 0.4,         
    'aug_saturation': 0.4,          
    'aug_hue': 0.2,                  
    
    # Normalizzazione
    'norm_mean': [0.485, 0.456, 0.406],
    'norm_std': [0.229, 0.224, 0.225],
    
    # Regolarizzazione
    'label_smoothing': 0.3,  
}

In [36]:
label_columns = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']
df = pd.read_csv(CONFIG['groundtruth_path'])
df['label'] = df[label_columns].idxmax(axis=1)

# Mappa da etichetta stringa a intero
label_map = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
df['label'] = df['label'].map(label_map)

print("Mappatura etichette:", label_map)

# Analizza distribuzione classi ORIGINALE
print("\nDistribuzione classi nel dataset originale:")
class_counts = df['label'].value_counts().sort_index()
for label_idx, count in class_counts.items():
    label_name = [k for k, v in label_map.items() if v == label_idx][0]
    print(f"{label_name} (classe {label_idx}): {count} campioni")
print(f"Totale campioni: {len(df)}")

# SPLIT PRIMA dell'oversampling per evitare data leakage
from sklearn.model_selection import train_test_split

# Primo split: train vs temp (val+test)
X = df['image'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Secondo split: val vs test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Crea DataFrame per ogni split
train_df = df[df['image'].isin(X_train)].copy()
val_df = df[df['image'].isin(X_val)].copy()
test_df = df[df['image'].isin(X_test)].copy()

print(f"\nSplit dataset:")
print(f"Training: {len(train_df)} campioni")
print(f"Validation: {len(val_df)} campioni") 
print(f"Test: {len(test_df)} campioni")

# Analizza distribuzione nel training set
print(f"\nDistribuzione classi nel TRAINING set (prima dell'oversampling):")
train_counts = train_df['label'].value_counts().sort_index()
for label_idx, count in train_counts.items():
    label_name = [k for k, v in label_map.items() if v == label_idx][0]
    print(f"{label_name} (classe {label_idx}): {count} campioni")

Mappatura etichette: {'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}

Distribuzione classi nel dataset originale:
AK (classe 0): 867 campioni
BCC (classe 1): 3323 campioni
BKL (classe 2): 2624 campioni
DF (classe 3): 239 campioni
MEL (classe 4): 4522 campioni
NV (classe 5): 12875 campioni
SCC (classe 6): 628 campioni
VASC (classe 7): 253 campioni
Totale campioni: 25331

Split dataset:
Training: 20264 campioni
Validation: 2533 campioni
Test: 2534 campioni

Distribuzione classi nel TRAINING set (prima dell'oversampling):
AK (classe 0): 694 campioni
BCC (classe 1): 2658 campioni
BKL (classe 2): 2099 campioni
DF (classe 3): 191 campioni
MEL (classe 4): 3618 campioni
NV (classe 5): 10300 campioni
SCC (classe 6): 502 campioni
VASC (classe 7): 202 campioni


In [37]:
from PIL import Image, ImageEnhance, ImageFilter
import random

def create_augmented_image(image_path, dataset_path):
    """
    Crea una versione augmentata di un'immagine
    """
    # Trova il percorso corretto dell'immagine
    img_path_jpg = os.path.join(dataset_path, f"{image_path}.jpg")
    img_path_png = os.path.join(dataset_path, f"{image_path}.png")
    
    if os.path.exists(img_path_jpg):
        img = Image.open(img_path_jpg).convert("RGB")
    elif os.path.exists(img_path_png):
        img = Image.open(img_path_png).convert("RGB")
    else:
        raise FileNotFoundError(f"Immagine non trovata: {image_path}")
    
    # Applica augmentazioni casuali
    # Rotazione casuale
    if random.random() > 0.5:
        angle = random.uniform(-30, 30)
        img = img.rotate(angle, fillcolor=(0, 0, 0))
    
    # Flip orizzontale
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    
    # Flip verticale  
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    
    # Aggiustamenti colore
    if random.random() > 0.5:
        # Luminosità
        enhancer = ImageEnhance.Brightness(img)
        img = enhancer.enhance(random.uniform(0.8, 1.2))
        
        # Contrasto
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(random.uniform(0.8, 1.2))
        
        # Saturazione
        enhancer = ImageEnhance.Color(img)
        img = enhancer.enhance(random.uniform(0.8, 1.2))
    
    # Blur leggero (occasionale)
    if random.random() > 0.8:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.0)))
    
    return img

def perform_oversampling_train_only(train_df, dataset_path, target_samples_per_class=None):
    """
    Effettua oversampling SOLO sul training set
    """
    if target_samples_per_class is None:
        # Usa il numero di campioni della classe più numerosa NEL TRAINING SET
        target_samples_per_class = train_df['label'].value_counts().max()
    
    print(f"\nTarget campioni per classe nel training: {target_samples_per_class}")
    
    oversampled_data = []
    
    for class_idx in sorted(train_df['label'].unique()):
        class_data = train_df[train_df['label'] == class_idx].copy()
        current_count = len(class_data)
        
        label_name = [k for k, v in label_map.items() if v == class_idx][0]
        print(f"\nProcessando classe {label_name} (indice {class_idx}):")
        print(f"  Campioni training attuali: {current_count}")
        
        # Aggiungi tutti i campioni originali del training
        oversampled_data.append(class_data)
        
        if current_count < target_samples_per_class:
            samples_needed = target_samples_per_class - current_count
            print(f"  Campioni da generare: {samples_needed}")
            
            # Genera campioni augmentati
            augmented_rows = []
            
            for i in tqdm(range(samples_needed), desc=f"Augmenting {label_name}"):
                # Seleziona casualmente un campione esistente da questa classe NEL TRAINING
                source_row = class_data.sample(n=1).iloc[0]
                
                # Crea nuovo ID univoco per l'immagine augmentata
                new_id = f"{source_row['image']}_aug_{i}"
                
                # Crea una copia della riga con nuovo ID
                new_row = source_row.copy()
                new_row['image'] = new_id
                new_row['is_augmented'] = True  # Flag per identificare immagini augmentate
                
                augmented_rows.append(new_row)
            
            # Converti in DataFrame e aggiungi
            if augmented_rows:
                augmented_df = pd.DataFrame(augmented_rows)
                oversampled_data.append(augmented_df)
                print(f"  Campioni generati: {len(augmented_df)}")
        else:
            print(f"  Classe già bilanciata")
    
    # Combina tutti i dati del training
    balanced_train_df = pd.concat(oversampled_data, ignore_index=True)
    
    # Aggiungi flag per immagini originali
    balanced_train_df.loc[balanced_train_df['image'].str.contains('_aug_') == False, 'is_augmented'] = False
    
    return balanced_train_df

# Effettua l'oversampling SOLO sul training set
print("Inizio oversampling del SOLO training set...")
balanced_train_df = perform_oversampling_train_only(train_df, CONFIG['dataset_path'])

print(f"\nTraining set bilanciato creato!")
print(f"Training originale: {len(train_df)} campioni")
print(f"Training dopo oversampling: {len(balanced_train_df)} campioni")

# Verifica la nuova distribuzione NEL TRAINING
print("\nDistribuzione classi DOPO l'oversampling (SOLO training set):")
balanced_counts = balanced_train_df['label'].value_counts().sort_index()
for label_idx, count in balanced_counts.items():
    label_name = [k for k, v in label_map.items() if v == label_idx][0]
    original_count = train_df[train_df['label'] == label_idx].shape[0]
    augmented_count = count - original_count
    print(f"{label_name} (classe {label_idx}): {count} campioni ({original_count} originali + {augmented_count} augmentati)")

# Aggiungi flag is_augmented ai set val e test (tutte False)
val_df['is_augmented'] = False
test_df['is_augmented'] = False

print(f"\nConferma: Val e Test set contengono SOLO immagini originali")
print(f"Val set: {len(val_df)} campioni (tutti originali)")
print(f"Test set: {len(test_df)} campioni (tutti originali)")

Inizio oversampling del SOLO training set...

Target campioni per classe nel training: 10300

Processando classe AK (indice 0):
  Campioni training attuali: 694
  Campioni da generare: 9606


Augmenting AK: 100%|██████████| 9606/9606 [00:04<00:00, 1977.06it/s]


  Campioni generati: 9606

Processando classe BCC (indice 1):
  Campioni training attuali: 2658
  Campioni da generare: 7642


Augmenting BCC: 100%|██████████| 7642/7642 [00:04<00:00, 1903.51it/s]


  Campioni generati: 7642

Processando classe BKL (indice 2):
  Campioni training attuali: 2099
  Campioni da generare: 8201


...

  Campioni generati: 6682

Processando classe NV (indice 5):
  Campioni training attuali: 10300
  Classe già bilanciata

Processando classe SCC (indice 6):
  Campioni training attuali: 502
  Campioni da generare: 9798


Augmenting SCC: 100%|██████████| 9798/9798 [00:05<00:00, 1731.48it/s]


  Campioni generati: 9798

Processando classe VASC (indice 7):
  Campioni training attuali: 202
  Campioni da generare: 10098


Augmenting VASC: 100%|██████████| 10098/10098 [00:06<00:00, 1617.18it/s]


  Campioni generati: 10098

Training set bilanciato creato!
Training originale: 20264 campioni
Training dopo oversampling: 82400 campioni

Distribuzione classi DOPO l'oversampling (SOLO training set):
AK (classe 0): 10300 campioni (694 originali + 9606 augmentati)
BCC (classe 1): 10300 campioni (2658 originali + 7642 augmentati)
BKL (classe 2): 10300 campioni (2099 originali + 8201 augmentati)
DF (classe 3): 10300 campioni (191 originali + 10109 augmentati)
MEL (classe 4): 10300 campioni (3618 originali + 6682 augmentati)
NV (classe 5): 10300 campioni (10300 originali + 0 augmentati)
SCC (classe 6): 10300 campioni (502 originali + 9798 augmentati)
VASC (classe 7): 10300 campioni (202 originali + 10098 augmentati)

Conferma: Val e Test set contengono SOLO immagini originali
Val set: 2533 campioni (tutti originali)
Test set: 2534 campioni (tutti originali)


In [38]:
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.data = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_id = row['image']
        label = row['label']
        is_augmented = row.get('is_augmented', False)
        
        # Se è un'immagine augmentata, genera l'augmentazione al volo
        if is_augmented and '_aug_' in image_id:
            # Estrai l'ID originale
            original_id = image_id.split('_aug_')[0]
            
            # Trova immagine originale JPG o PNG
            img_path_jpg = os.path.join(self.image_dir, f"{original_id}.jpg")
            img_path_png = os.path.join(self.image_dir, f"{original_id}.png")
            
            if os.path.exists(img_path_jpg):
                image = create_augmented_image(original_id, self.image_dir)
            elif os.path.exists(img_path_png):
                image = create_augmented_image(original_id, self.image_dir)
            else:
                raise FileNotFoundError(f"File originale non trovato: {original_id}")
        else:
            # Carica immagine originale normalmente
            img_path_jpg = os.path.join(self.image_dir, f"{image_id}.jpg")
            img_path_png = os.path.join(self.image_dir, f"{image_id}.png")
            
            if os.path.exists(img_path_jpg):
                image = Image.open(img_path_jpg).convert("RGB")
            elif os.path.exists(img_path_png):
                image = Image.open(img_path_png).convert("RGB")
            else:
                raise FileNotFoundError(f"File non trovato: {image_id}")

        if self.transform:
            image = self.transform(image)

        return image, label

In [39]:
import math
from torch.utils.data import DataLoader

# Converti image size e batch size da stringhe a interi se necessario
image_size = int(CONFIG['image_size'])
batch_size = int(CONFIG['batch_size'])

# Trasformazioni usando parametri da CONFIG
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=CONFIG['aug_hflip_prob']),
    transforms.RandomVerticalFlip(p=CONFIG['aug_vflip_prob']),
    transforms.RandomRotation(CONFIG['aug_rotation']),
    transforms.RandomResizedCrop(image_size, scale=CONFIG['aug_crop_scale']),
    transforms.ColorJitter(
        brightness=CONFIG['aug_brightness'], 
        contrast=CONFIG['aug_contrast'], 
        saturation=CONFIG['aug_saturation'], 
        hue=CONFIG['aug_hue']
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG['norm_mean'], std=CONFIG['norm_std'])
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG['norm_mean'], std=CONFIG['norm_std'])
])

# Crea dataset separati per train/val/test
train_dataset = SkinLesionDataset(balanced_train_df, CONFIG['dataset_path'], transform=train_transform)
val_dataset = SkinLesionDataset(val_df, CONFIG['dataset_path'], transform=eval_transform)
test_dataset = SkinLesionDataset(test_df, CONFIG['dataset_path'], transform=eval_transform)

# Crea i DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train set: {len(train_dataset)} campioni (con oversampling)")
print(f"Val set: {len(val_dataset)} campioni (solo originali)")
print(f"Test set: {len(test_dataset)} campioni (solo originali)")

Train set: 82400 campioni (con oversampling)
Val set: 2533 campioni (solo originali)
Test set: 2534 campioni (solo originali)


In [40]:
import timm
import torch.nn as nn
import torch
from torch.cuda.amp import GradScaler
import math

# Numero di classi nel dataset
num_classes = len(label_map)

# Crea modello EfficientNet ottimizzato
model = timm.create_model(
    CONFIG['modello'],
    pretrained=True,
    num_classes=num_classes,
    drop_rate=CONFIG['dropout_rate'],
    drop_path_rate=CONFIG['drop_path_rate'],
)

# Sposta su GPU se disponibile
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Setup Mixed Precision se abilitato
scaler = GradScaler() if CONFIG['mixed_precision'] else None

print(f"Modello {CONFIG['modello']} caricato su {device}")
print(f"Parametri totali: {sum(p.numel() for p in model.parameters()):,}")
print(f"Parametri trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Dropout rate: {CONFIG['dropout_rate']}")
print(f"Drop path rate: {CONFIG['drop_path_rate']}")
print(f"Mixed precision: {CONFIG['mixed_precision']}")

# Verifica memoria GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    memory_allocated = torch.cuda.memory_allocated() / 1e9
    memory_reserved = torch.cuda.memory_reserved() / 1e9
    print(f"Memoria GPU allocata: {memory_allocated:.2f} GB")
    print(f"Memoria GPU riservata: {memory_reserved:.2f} GB")

Modello efficientnet_b4 caricato su cuda
Parametri totali: 17,562,960
Parametri trainable: 17,562,960
Dropout rate: 0.7
Drop path rate: 0.3
Mixed precision: True
Memoria GPU allocata: 0.40 GB
Memoria GPU riservata: 2.80 GB


In [41]:
import torch.optim as optim
import math

# Focal Loss con Label Smoothing opzionale
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, label_smoothing=0.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        if self.label_smoothing > 0:
            # Implementa label smoothing
            num_classes = inputs.size(-1)
            smoothed_targets = torch.zeros_like(inputs).scatter_(
                1, targets.unsqueeze(1), 1.0 - self.label_smoothing
            )
            smoothed_targets += self.label_smoothing / num_classes
            
            ce_loss = -(smoothed_targets * torch.log_softmax(inputs, dim=1)).sum(dim=1)
        else:
            ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return torch.mean(focal_loss)

# Setup Loss Function
if CONFIG['loss_type'] == 'focal':
    criterion = FocalLoss(
        alpha=CONFIG['focal_alpha'], 
        gamma=CONFIG['focal_gamma'],
        label_smoothing=CONFIG.get('label_smoothing', 0.0)
    )
    print(f"Focal Loss con α={CONFIG['focal_alpha']}, γ={CONFIG['focal_gamma']}")
    if CONFIG.get('label_smoothing', 0.0) > 0:
        print(f"Label smoothing: {CONFIG['label_smoothing']}")
else:
    if CONFIG.get('label_smoothing', 0.0) > 0:
        criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
        print(f"CrossEntropy Loss con label smoothing: {CONFIG['label_smoothing']}")
    else:
        criterion = nn.CrossEntropyLoss()
        print("CrossEntropy Loss standard")

# Ottimizzatore AdamW ottimizzato
optimizer = optim.AdamW(
    model.parameters(), 
    lr=CONFIG['LR'], 
    weight_decay=CONFIG['WD'],
    betas=(0.9, 0.999),
    eps=1e-8
)

# Setup Scheduler avanzato
total_steps = len(train_loader) * CONFIG['epoche']
warmup_steps = len(train_loader) * CONFIG.get('warmup_epochs', 5)

if CONFIG['scheduler_type'] == 'cosine':
    # Cosine Annealing con Warmup
    def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, min_lr_ratio=0.01):
        def lr_lambda(current_step):
            if current_step < num_warmup_steps:
                return float(current_step) / float(max(1, num_warmup_steps))
            
            progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
            cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
            return max(min_lr_ratio, cosine_decay)
        
        return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, 
        warmup_steps, 
        total_steps, 
        min_lr_ratio=CONFIG.get('min_lr', 1e-6) / CONFIG['LR']
    )
    scheduler_step_type = 'step'  # Chiamare ad ogni batch
    print(f"Cosine scheduler con warmup ({CONFIG.get('warmup_epochs', 5)} epoche)")
    
elif CONFIG['scheduler_type'] == 'plateau':
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=CONFIG['scheduler_factor'], 
        patience=CONFIG['scheduler_patience'], 
        verbose=True,
        min_lr=CONFIG.get('min_lr', 1e-6)
    )
    scheduler_step_type = 'epoch'  # Chiamare ad ogni epoca
    print(f"ReduceLROnPlateau scheduler (factor={CONFIG['scheduler_factor']}, patience={CONFIG['scheduler_patience']})")
    
else:
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, 
        step_size=10, 
        gamma=0.5
    )
    scheduler_step_type = 'epoch'
    print("StepLR scheduler")

print(f"Learning rate: {CONFIG['LR']}")
print(f"Weight decay: {CONFIG['WD']}")
print(f"Epoche totali: {CONFIG['epoche']}")
print(f"Steps totali: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

Focal Loss con α=1.0, γ=2.0
Label smoothing: 0.3
Cosine scheduler con warmup (3 epoche)
Learning rate: 3e-05
Weight decay: 0.003
Epoche totali: 25
Steps totali: 206,000
Warmup steps: 24,720


In [ ]:
from torch.cuda.amp import autocast

def train_one_epoch(model, dataloader, optimizer, criterion, device, scaler=None, scheduler=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress = tqdm(dataloader, desc="Training", leave=False)

    for batch_idx, (inputs, labels) in enumerate(progress):
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        
        # Mixed precision training
        if scaler is not None:
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG['grad_clip_norm'])
            
            # Memorizza la scala prima dell'update
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            
            # Scheduler step solo se l'optimizer ha davvero fatto un passo
            # (la scala non dovrebbe diminuire se tutto va bene)
            if scheduler is not None and CONFIG['scheduler_type'] == 'cosine':
                # Se la nuova scala è >= della vecchia, l'optimizer step è andato a buon fine
                if scaler.get_scale() >= old_scale * 0.99:  # Piccola tolleranza per errori floating point
                    scheduler.step()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG['grad_clip_norm'])
            optimizer.step()
            
            # Per training senza mixed precision, scheduler step sempre dopo optimizer step
            if scheduler is not None and CONFIG['scheduler_type'] == 'cosine':
                scheduler.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        avg_loss = running_loss / total
        avg_acc = correct / total
        progress.set_description(f"Training | Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        progress = tqdm(dataloader, desc="Evaluating", leave=False)
        
        for inputs, labels in progress:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            # Mixed precision per evaluation
            if CONFIG['mixed_precision']:
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            avg_loss = running_loss / total
            avg_acc = correct / total
            progress.set_description(f"Evaluating | Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Inizializza liste per metriche
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
epochs_list = []

n_epochs = CONFIG['epoche']

print(f"Inizio training per {n_epochs} epoche")
print(f"Batch size: {CONFIG['batch_size']}, Image size: {CONFIG['image_size']}")
print(f"Modello: {CONFIG['modello']}")
print(f"Mixed precision: {CONFIG['mixed_precision']}")
print(f"Scheduler: {CONFIG['scheduler_type']}")
print(f"Label smoothing: {CONFIG.get('label_smoothing', 0.0)}")
print(f"Steps totali: {len(train_loader) * n_epochs:,}")

best_val_acc = 0.0
patience_counter = 0

for epoch in range(n_epochs):
    print(f"\nEpoch {epoch+1}/{n_epochs}")
    
    # Training con mixed precision e scheduler cosine 
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler, 
        scheduler if CONFIG['scheduler_type'] == 'cosine' else None
    )
    
    # Validation
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    # Scheduler step per epoch se non cosine (DOPO train_one_epoch)
    if CONFIG['scheduler_type'] == 'plateau':
        scheduler.step(val_loss)
    elif CONFIG['scheduler_type'] != 'cosine':
        scheduler.step()
    
    # Salva miglior modello
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if hasattr(scheduler, 'state_dict') else None,
            'val_acc': val_acc,
            'train_acc': train_acc,
            'config': CONFIG
        }, 'best_model_efficientnet_b4.pth')
        print(f"Nuovo miglior modello salvato! Val Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
    
    # Salva metriche
    epochs_list.append(epoch + 1)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    # Stampa risultati dettagliati
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    print(f"LR    | {current_lr:.2e}")
    print(f"Best Val Acc: {best_val_acc:.4f}")


print(f"\nTraining completato")
print(f"Migliore validation accuracy: {best_val_acc:.4f}")
print(f"Modello salvato come: best_model_efficientnet_b4.pth")

Inizio training per 25 epoche
Batch size: 10, Image size: 320
Modello: efficientnet_b4
Mixed precision: True
Scheduler: cosine
Label smoothing: 0.3
Steps totali: 206,000

Epoch 1/25


Nuovo miglior modello salvato! Val Acc: 0.2795
Train | Loss: 2.1544 | Acc: 0.1447
Val   | Loss: 1.4930 | Acc: 0.2795
LR    | 9.99e-06
Best Val Acc: 0.2795

Epoch 2/25


Nuovo miglior modello salvato! Val Acc: 0.5570
Train | Loss: 1.5269 | Acc: 0.2847
Val   | Loss: 1.1315 | Acc: 0.5570
LR    | 2.00e-05
Best Val Acc: 0.5570

Epoch 3/25


Training | Loss: 1.2923 | Acc: 0.4373:  59%|█████▉    | 4858/8240 [08:54<05:57,  9.46it/s]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Grafico Loss
ax1.plot(epochs_list, train_losses, 'b-o', label='Training Loss', linewidth=2, markersize=6)
ax1.plot(epochs_list, val_losses, 'r-s', label='Validation Loss', linewidth=2, markersize=6)
ax1.set_title('📉 Loss durante il Training', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(epochs_list)

# Grafico Accuracy
ax2.plot(epochs_list, [acc*100 for acc in train_accuracies], 'b-o', label='Training Accuracy', linewidth=2, markersize=6)
ax2.plot(epochs_list, [acc*100 for acc in val_accuracies], 'r-s', label='Validation Accuracy', linewidth=2, markersize=6)
ax2.set_title('📈 Accuracy durante il Training', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epochs_list)
ax2.set_ylim([0, 100])

plt.tight_layout()
plt.show()

# Stampa statistiche finali
print(f"\nRISULTATI FINALI:")
print(f"Migliore Train Accuracy: {max(train_accuracies):.4f} (Epoch {train_accuracies.index(max(train_accuracies))+1})")
print(f"Migliore Val Accuracy: {max(val_accuracies):.4f} (Epoch {val_accuracies.index(max(val_accuracies))+1})")
print(f"Migliore Train Loss: {min(train_losses):.4f} (Epoch {train_losses.index(min(train_losses))+1})")
print(f"Migliore Val Loss: {min(val_losses):.4f} (Epoch {val_losses.index(min(val_losses))+1})")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Report e Confusion Matrix
print("Accuracy test:", np.mean(np.array(y_true) == np.array(y_pred)))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_map.keys()))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_map.keys())
disp.plot(xticks_rotation=45, cmap="Blues")
plt.title("Confusion Matrix")
plt.show()
